# SAMS Standards — Clause Chunker

Extracts each numbered clause (e.g. `7.2.1`, `7.3`, `7.5`) from `sams_standard.pdf` as an isolated, labelled chunk.

**Strategy**:
- `pdfplumber` reads character-level coordinates + font metadata
- Regex `^\d{1,3}(\.\d+)+$` identifies clause numbers
- Auto-detects TOC pages (skips them) and the clause-number column via x0 statistics

**Outputs**: `sams_chunks.json`, `sams_chunks.csv`, `sams_chunks.xlsx`

---
## 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys

for pkg in ['pdfplumber', 'pymupdf', 'pandas', 'openpyxl']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All dependencies installed.')

In [10]:
import pdfplumber
import fitz  # PyMuPDF — available for font inspection if needed
import pandas as pd
import json, re, statistics
from pathlib import Path
from collections import defaultdict
from IPython.display import display

# ── Config ────────────────────────────────────────
PDF_PATH  = Path(r'd:\Machine Learning\JFF JOB\Engineering_drawing_chunking\sams_standard.pdf')
OUT_JSON  = PDF_PATH.parent / 'sams_chunks.json'
OUT_CSV   = PDF_PATH.parent / 'sams_chunks.csv'
OUT_XLSX  = PDF_PATH.parent / 'sams_chunks.xlsx'

assert PDF_PATH.exists(), f'PDF not found: {PDF_PATH}'
print(f'PDF: {PDF_PATH.name}  ({PDF_PATH.stat().st_size / 1024:.1f} KB)')

PDF: sams_standard.pdf  (823.0 KB)


---
## 2 — PDF Explorer: Page Count & Raw Text Preview

In [11]:
with pdfplumber.open(PDF_PATH) as pdf:
    total_pages = len(pdf.pages)
    print(f'Total pages: {total_pages}')
    
    # Preview the first page that looks like body content (skip cover pages)
    for pg_idx in range(total_pages):
        text = pdf.pages[pg_idx].extract_text() or ''
        if re.search(r'\d+\.\d+', text):
            print(f'\n──── First clause-containing page: {pg_idx+1} ────')
            for i, line in enumerate(text.split('\n')[:40]):
                print(f'{i+1:3d} | {line}')
            break

Total pages: 48

──── First clause-containing page: 3 ────
  1 | Document Responsibility: Vessels Standards Committee 32-SAMSS-004
  2 | Issue Date: 11 July 2023 Manufacture of Pressure Vessels
  3 | Next Revision: 11 July 2028
  4 | Summary of Changes
  5 | Paragraph Number
  6 | Change Type
  7 | Technical Change(s)
  8 | Previous Revision Current Revision
  9 | (Addition, Modification, Deletion,
 10 | 24 June 2020 11 July 2023 New)
 11 | Global Global Modification ASME modified to ASME
 12 | BPVC
 13 | - 1.3.10) Addition Paragraph added
 14 | 3.1 3.1 Addition SAES-W-014 added.
 15 | 3.2 3.2 Addition ASCE 7-10 added.
 16 | 4 4 Deletion Design thickness deleted.
 17 | 7.2.3 7.2.3 Modification Paragraph modified.
 18 | 7.3.2 7.3.2 Modification Paragraph modified.
 19 | 7.9.1. 7.9.1 Modification Paragraph modified.
 20 | 7.10.5 7.10.5 Modification Paragraph modified.
 21 | 7.11.1.b) 7.11.1.b) Modification Paragraph modified.
 22 | 7.11.7 7.11.7 Modification Paragraph modified.
 23 | 8.5

---
## 3 — Font & Position Analysis

Identifies the x0 column where clause numbers appear, and automatically skips TOC pages (where clause IDs appear in two columns).

In [12]:
CLAUSE_RE = re.compile(r'^\d{1,3}(\.\d+)+$')  # matches 7.2.1, 10.3, 7.11.7 etc.

# ── Clause-number column zone ─────────────────────────────────────────
# Analysis of the PDF confirmed the left clause column is at x0 ≈ 36pt.
# Cross-references inside body text / tables appear at x0 > 100pt and are excluded.
# This single tight zone replaces complex TOC-page detection, which was too fragile.
CLAUSE_X0_MIN = 25   # left boundary
CLAUSE_X0_MAX = 65   # right boundary (32-SAMSS clause numbers are short: '7.11.7')

# No pages are skipped — all 48 pages are processed; the x0 zone does the filtering.
toc_pages = set()  # kept for API compatibility with chunk_pdf()

with pdfplumber.open(PDF_PATH) as pdf:
    total_pages = len(pdf.pages)
print(f'Clause x0 detection zone : [{CLAUSE_X0_MIN}, {CLAUSE_X0_MAX}] pt')
print(f'Pages to process          : {total_pages} (all pages, no skipping)')

Clause x0 detection zone : [25, 65] pt
Pages to process          : 48 (all pages, no skipping)


---
## 4 — Core Helper Functions

In [13]:
def extract_line_words(page):
    """
    Returns words grouped into logical lines (by y-position bucket).
    Each line is a list of word dicts, sorted left-to-right.
    """
    words = page.extract_words(
        extra_attrs=['fontname', 'size'],
        keep_blank_chars=False,
        x_tolerance=3,
        y_tolerance=3,
    )
    if not words:
        return []
    lines = defaultdict(list)
    for w in words:
        key = round(float(w['top']) / 3) * 3  # group within 3pt vertical bucket
        lines[key].append(w)
    return [
        sorted(lines[k], key=lambda w: w['x0'])
        for k in sorted(lines.keys())
    ]


def is_clause_line(line_words, x0_min, x0_max):
    """
    Returns (clause_id, rest_text) if this line starts a new clause.
    Returns (None, None) otherwise.
    """
    if not line_words:
        return None, None
    first = line_words[0]
    txt   = first['text'].strip()
    x0    = float(first['x0'])
    if CLAUSE_RE.match(txt) and x0_min <= x0 <= x0_max:
        rest = ' '.join(w['text'] for w in line_words[1:]).strip()
        return txt, rest
    return None, None

print('Helper functions defined.')

Helper functions defined.


---
## 5 — Full-Document Chunker

Walks every body page. Each new clause number starts a fresh chunk; text is accumulated until the next clause.

In [14]:
def chunk_pdf(pdf_path, toc_pages_set, x0_min, x0_max):
    """
    Returns list of clause dicts:
      clause_id, text, page_start, page_end, depth, parent, section
    """
    chunks  = []
    current = None   # { id, raw, ps, pe }

    def flush(c):
        if c is None:
            return
        text = c['raw'].strip()
        if not text:
            return
        parts = c['id'].split('.')
        chunks.append({
            'clause_id':  c['id'],
            'text':       text,
            'page_start': c['ps'],
            'page_end':   c['pe'],
            'depth':      len(parts),
            'parent':     '.'.join(parts[:-1]) if len(parts) > 1 else None,
            'section':    parts[0],
        })

    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        print(f'Chunking {total} pages  (skipping TOC pages: {sorted(toc_pages_set)})...')

        for pg_idx, page in enumerate(pdf.pages):
            pg_num = pg_idx + 1
            if pg_num in toc_pages_set:
                continue
            if pg_num % 10 == 0 or pg_num == total:
                print(f'  Page {pg_num}/{total}  →  chunks so far: {len(chunks)}')

            for line_words in extract_line_words(page):
                clause_id, rest = is_clause_line(line_words, x0_min, x0_max)
                if clause_id:
                    flush(current)
                    current = {'id': clause_id, 'raw': rest + ' ', 'ps': pg_num, 'pe': pg_num}
                elif current:
                    lt = ' '.join(w['text'] for w in line_words).strip()
                    if lt:
                        current['raw'] += lt + ' '
                        current['pe'] = pg_num

        flush(current)

    return chunks

print('chunk_pdf() defined.')

chunk_pdf() defined.


In [15]:
# ── Run ────────────────────────────────────────────
chunks_raw = chunk_pdf(PDF_PATH, toc_pages, CLAUSE_X0_MIN, CLAUSE_X0_MAX)

# Deduplicate: if the same clause_id appears on multiple pages (e.g. continued),
# keep the entry with the longest text (most content).
merged = {}
for c in chunks_raw:
    cid = c['clause_id']
    if cid not in merged or len(c["text"]) > len(merged[cid]["text"]):
        merged[cid] = c
chunks = list(merged.values())

print(f'\n✅  Raw chunks extracted  : {len(chunks_raw)}')
print(f'✅  Unique chunks (deduped): {len(chunks)}')

Chunking 48 pages  (skipping TOC pages: [])...
  Page 10/48  →  chunks so far: 15
  Page 20/48  →  chunks so far: 105
  Page 30/48  →  chunks so far: 190
  Page 40/48  →  chunks so far: 251
  Page 48/48  →  chunks so far: 288

✅  Raw chunks extracted  : 289
✅  Unique chunks (deduped): 283


---
## 6 — Sort & Hierarchy Check

In [16]:
# Numerically sort by clause ID (so 7.10.1 > 7.9.1)
chunks.sort(key=lambda c: [int(p) for p in c['clause_id'].split('.')])

# Build index for parent resolution check
id_set = {c['clause_id'] for c in chunks}
orphans = [c['clause_id'] for c in chunks if c['parent'] and c['parent'] not in id_set]

print(f'Chunks sorted.')
print(f'Top-level sections: {sorted(set(c["section"] for c in chunks), key=int)}')

if orphans:
    print(f'\n⚠  Orphan clauses (parent not found): {orphans[:15]}')
    print('   These may be sub-clauses of section headers that appear in bold without a sub-number.')
else:
    print('\n✅  All parent references resolved.')

Chunks sorted.
Top-level sections: ['3', '7', '8', '10', '11', '12', '13', '14', '16', '19']

⚠  Orphan clauses (parent not found): ['3.1', '3.2', '7.1.1', '7.1.2', '7.1.3', '7.1.4', '7.1.5', '7.1.6', '7.1.7', '7.1.8', '7.2.1', '7.2.2', '7.2.3', '7.2.4', '7.2.5']
   These may be sub-clauses of section headers that appear in bold without a sub-number.


---
## 7 — Export to JSON / CSV / Excel

In [17]:
# JSON
with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print(f'✅  JSON  → {OUT_JSON}')

# CSV & Excel
df = pd.DataFrame(chunks)
df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f'✅  CSV   → {OUT_CSV}')
df.to_excel(OUT_XLSX, index=False)
print(f'✅  Excel → {OUT_XLSX}')

print(f'\nDataFrame: {df.shape[0]} rows × {df.shape[1]} cols')
print(df.columns.tolist())

✅  JSON  → d:\Machine Learning\JFF JOB\Engineering_drawing_chunking\sams_chunks.json
✅  CSV   → d:\Machine Learning\JFF JOB\Engineering_drawing_chunking\sams_chunks.csv
✅  Excel → d:\Machine Learning\JFF JOB\Engineering_drawing_chunking\sams_chunks.xlsx

DataFrame: 283 rows × 7 cols
['clause_id', 'text', 'page_start', 'page_end', 'depth', 'parent', 'section']


---
## 8 — QA: Visual Verification

In [ ]:
# ── Summary stats ──────────────────────────────────
print('=== EXTRACTION SUMMARY ===')
print(f'Total chunks : {len(chunks)}')

print('\nDepth distribution (1=section, 2=subsection, 3=clause, 4=subclause):')
display(df['depth'].value_counts().sort_index().rename('count'))

print('\nSection distribution:')
display(df.groupby('section')['clause_id'].count().rename('clause_count'))

In [ ]:
# ── Preview table ──────────────────────────────────
df_preview = df[['clause_id', 'depth', 'parent', 'page_start', 'text']].copy()
df_preview['text'] = df_preview['text'].str[:100] + '…'

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 200)
display(df_preview.head(40))

In [ ]:
# ── Drill into a specific clause ───────────────────
TARGET_CLAUSE = '7.2.1'  # ← change to any clause you want to verify

result = next((c for c in chunks if c['clause_id'] == TARGET_CLAUSE), None)
if result:
    print(f'Clause  : {result["clause_id"]}')
    print(f'Pages   : {result["page_start"]} – {result["page_end"]}')
    print(f'Depth   : {result["depth"]}   Parent: {result["parent"]}   Section: {result["section"]}')
    print(f'\nText:\n{result["text"]}')
else:
    print(f'Clause {TARGET_CLAUSE} not found.')
    print('Available IDs (first 30):', [c['clause_id'] for c in chunks[:30]])

In [ ]:
# ── Section tree view ─────────────────────────────
SECTION_TO_SHOW = '7'  # ← change to any top-level section

section_chunks = [c for c in chunks if c['section'] == SECTION_TO_SHOW]
print(f'Section {SECTION_TO_SHOW} — {len(section_chunks)} clauses\n')
for c in section_chunks:
    indent  = '  ' * (c['depth'] - 1)
    preview = c['text'][:90].replace('\n', ' ')
    print(f"{indent}{c['clause_id']:15s}  {preview}…")

---
## 9 — Troubleshooting & Fine-Tuning

Use this if clauses were missed or incorrect text was captured.

In [ ]:
# Inspect raw word positions on any page
DEBUG_PAGE = 10  # 1-indexed

with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[DEBUG_PAGE - 1]
    words = page.extract_words(extra_attrs=['fontname', 'size'], x_tolerance=3, y_tolerance=3)

df_debug = pd.DataFrame([{
    'text':       w['text'],
    'x0':         round(w['x0'], 1),
    'top':        round(w['top'], 1),
    'fontname':   w.get('fontname', ''),
    'size':       round(float(w.get('size', 0)), 1),
    'is_clause?': bool(CLAUSE_RE.match(w['text'].strip()))
} for w in words])

print(f'Clause words on page {DEBUG_PAGE}:')
display(df_debug[df_debug['is_clause?']].head(20))

In [ ]:
# ── Re-run with manual x0 override if needed ──────
# Uncomment and adjust these two lines:
# CLAUSE_X0_MIN = 30
# CLAUSE_X0_MAX = 85
# chunks = chunk_pdf(PDF_PATH, toc_pages, CLAUSE_X0_MIN, CLAUSE_X0_MAX)
# print(f'Re-run: {len(chunks)} chunks')